In [ ]:
# =============================================================================
# Federated Learning: FedAvg on PlantVillage Dataset
# Conditions: 5 clients, Dirichlet(α=0.5), ResNet18, SGD(lr=0.001, mom=0.9),
#             batch=32, 5 local epochs, 10 rounds
# =============================================================================

import os, copy, json, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# =============================================================================
# CONFIG
# =============================================================================
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLIENTS  = 5
ALPHA        = 0.5
NUM_ROUNDS   = 10
LOCAL_EPOCHS = 5
BATCH_SIZE   = 32
LR           = 0.001
MOMENTUM     = 0.9
OUTPUT_DIR   = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Using device: {DEVICE}")

# =============================================================================
# AUTO-DETECT DATASET ROOT
# =============================================================================
def is_valid_imagefolder_root(path):
    """
    A valid ImageFolder root has at least 2 subdirectories,
    and those subdirectories contain image files (not more subdirectories).
    """
    p = Path(path)
    if not p.exists():
        return False
    subdirs = [x for x in p.iterdir() if x.is_dir()]
    if len(subdirs) < 2:
        return False
    # Check that at least one subdir contains image files directly
    for sd in subdirs[:5]:  # sample first 5
        imgs = [f for f in sd.iterdir()
                if f.is_file() and f.suffix.lower() in (".jpg", ".jpeg", ".png")]
        if imgs:
            return True
    return False

def find_root():
    # Priority candidates (color split preferred over segmented/grayscale)
    priority = [
        "/kaggle/input/plantvillage-dataset/plantvillage dataset/color",
        "/kaggle/input/plantvillage-dataset/color",
        "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color",
        "/kaggle/input/plantvillage-dataset/plantvillage dataset/segmented",
        "/kaggle/input/plantvillage-dataset/segmented",
        "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented",
        "/kaggle/input/plantvillage-dataset/plantvillage dataset/grayscale",
        "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/grayscale",
        "/kaggle/input/plantvillage-dataset",
        "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset",
    ]
    for p in priority:
        if is_valid_imagefolder_root(p):
            return p

    # Fallback: walk and find the first folder that qualifies
    for root, dirs, files in os.walk("/kaggle/input"):
        if is_valid_imagefolder_root(root):
            return root

    raise FileNotFoundError(
        "Could not find PlantVillage dataset. "
        "Make sure you added the dataset to this notebook."
    )

DATA_ROOT = find_root()
print(f"[INFO] Dataset root: {DATA_ROOT}")

# =============================================================================
# DATASET & PARTITION
# =============================================================================
TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(DATA_ROOT, transform=TRANSFORM)
NUM_CLASSES  = len(full_dataset.classes)
print(f"[INFO] Classes: {NUM_CLASSES}  |  Total images: {len(full_dataset)}")

def dirichlet_partition(dataset, num_clients, alpha, seed=42):
    rng         = np.random.default_rng(seed)
    labels      = np.array(dataset.targets)
    num_classes = len(dataset.classes)

    class_indices = defaultdict(list)
    for idx, lbl in enumerate(labels):
        class_indices[lbl].append(idx)

    client_indices = defaultdict(list)
    for c in range(num_classes):
        idxs = np.array(class_indices[c])
        rng.shuffle(idxs)
        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        splits = (proportions / proportions.sum() * len(idxs)).astype(int)
        diff = len(idxs) - splits.sum()
        for i in range(abs(diff)):
            splits[i % num_clients] += 1 if diff > 0 else -1
        ptr = 0
        for cid, s in enumerate(splits):
            client_indices[cid].extend(idxs[ptr:ptr + s].tolist())
            ptr += s
    return client_indices

client_indices = dirichlet_partition(full_dataset, NUM_CLIENTS, ALPHA)

client_train, client_test = {}, {}
for cid, indices in client_indices.items():
    random.shuffle(indices)
    n_train = int(0.8 * len(indices))
    client_train[cid] = indices[:n_train]
    client_test[cid]  = indices[n_train:]
    print(f"  Client {cid}: train={len(client_train[cid])}  test={len(client_test[cid])}")

def get_loader(indices, shuffle=True):
    return DataLoader(
        Subset(full_dataset, indices),
        batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=2, pin_memory=(DEVICE.type == "cuda")
    )

# =============================================================================
# MODEL
# =============================================================================
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)

# =============================================================================
# EVALUATION
# =============================================================================
def evaluate_global(model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for cid in range(NUM_CLIENTS):
            for imgs, lbls in get_loader(client_test[cid], shuffle=False):
                imgs = imgs.to(DEVICE)
                preds = model(imgs).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(lbls.numpy())
    return {
        "accuracy":  accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall":    recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1":        f1_score(all_labels, all_preds, average="macro", zero_division=0),
    }

# =============================================================================
# FedAvg
# =============================================================================
def local_train(global_state, client_id):
    model = build_model()
    model.load_state_dict(copy.deepcopy(global_state))
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    criterion = nn.CrossEntropyLoss()
    for _ in range(LOCAL_EPOCHS):
        for imgs, lbls in get_loader(client_train[client_id]):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(imgs), lbls).backward()
            optimizer.step()
    # return state on CPU to free GPU memory
    return {k: v.cpu() for k, v in model.state_dict().items()}, len(client_train[client_id])

def aggregate(updates):
    total     = sum(n for _, n in updates)
    new_state = copy.deepcopy(updates[0][0])
    for key in new_state:
        new_state[key] = sum(s[key].float() * (n / total) for s, n in updates)
    return new_state

# =============================================================================
# MAIN
# =============================================================================
global_model = build_model()
global_state = {k: v.cpu() for k, v in global_model.state_dict().items()}
results = []

print("\n" + "="*60)
print("FEDAVG")
print("="*60)

for rnd in range(1, NUM_ROUNDS + 1):
    updates = [local_train(global_state, cid) for cid in range(NUM_CLIENTS)]
    global_state = aggregate(updates)
    global_model.load_state_dict({k: v.to(DEVICE) for k, v in global_state.items()})

    metrics = evaluate_global(global_model)
    metrics["round"] = rnd
    results.append(metrics)

    print(f"  Round {rnd:2d} | Acc={metrics['accuracy']:.4f} | "
          f"P={metrics['precision']:.4f} | R={metrics['recall']:.4f} | "
          f"F1={metrics['f1']:.4f}")

    # save immediately after every round
    torch.save(global_state, OUTPUT_DIR / f"fedavg_round{rnd:02d}.pt")
    pd.DataFrame(results).to_csv(OUTPUT_DIR / "fedavg_metrics.csv", index=False)
    with open(OUTPUT_DIR / "fedavg_metrics.json", "w") as f:
        json.dump(results, f, indent=2)

print(f"\n[Done] All outputs saved to {OUTPUT_DIR}")